# 045 — Series temporales y backtesting

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Estructura:** `y_t = T_t + S_t + e_t` (o multiplicativa; log la vuelve aditiva).
**Estacionariedad** (media/varianza/autocovarianza constantes) es el supuesto de los
modelos clásicos; se alcanza diferenciando (`y_t − y_{t−1}`; estacional `y_t − y_{t−s}`).

**ARIMA(p,d,q):** AR = regresión sobre p rezagos propios; I = d diferencias; MA = q
errores pasados. ARIMA(0,1,0) = paseo aleatorio = baseline naive (`ŷ_t = y_{t−1}`).
Metodología Box-Jenkins: identificar con ACF/PACF → estimar → residuos ruido blanco.

**Backtesting (rolling origin):** entrenar con el pasado, predecir el bloque siguiente,
avanzar el origen y repetir (ventana expansiva o deslizante). Nunca split aleatorio:
fuga temporal. Features (rezagos, medias móviles) calculadas solo con datos ≤ t.

**Métrica honesta:** MAE/RMSE por horizonte + skill contra naive:
`skill = 1 − MAE_modelo/MAE_naive`. Sin superar al naive, no hay modelo.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Ejercicios

**Ejercicio 1 — Naive vs. media móvil.** Serie: [8, 10, 9, 11, 10, 12, 11]. Con backtest
de horizonte 1 sobre t = 4..7, calcula el MAE del naive (`ŷ_t = y_{t−1}`) y el de la media
móvil de 3. Calcula el skill de la MM3 sobre el naive. ¿Qué patrón de la serie explica el
resultado?

**Ejercicio 2 — Diferenciación y estacionariedad.** Toma la serie
y = [5, 8, 11, 14, 17, 20]. (a) ¿Es estacionaria? ¿Por qué? (b) Calcula la primera
diferencia y'. (c) ¿Qué modelo trivial genera y, y qué ARIMA(p,d,q) le corresponde?

**Ejercicio 3 — Detectar la fuga temporal.** Un colega crea la feature "media móvil
centrada de 7 días" (3 días antes + día + 3 después), hace split aleatorio 80/20 y
reporta MAE espectacular. Señala las DOS fugas distintas de su pipeline y cómo corregir
cada una.

**Ejercicio 4 — Folds de walk-forward.** Con 24 meses de datos, horizonte de test de
3 meses y ventana expansiva empezando con 12 meses de train, escribe los folds
(rangos train → test) del backtest. ¿Cuántos folds salen y por qué el último termina en
el mes 24?


In [ ]:
# TODO: ejecuta run_lab("ml", seed=45)
# TODO: comprueba que el resultado incluya las claves 'kind' y 'evidence'
result = None


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicio 1: backtest naive vs. media móvil (h=1)
y = [8, 10, 9, 11, 10, 12, 11]   # índices 0..6; predecimos t = 3..6 (0-based)

def backtest_h1(y, prediccion):
    errores = []
    for t in range(3, len(y)):
        errores.append(abs(y[t] - prediccion(y, t)))
    return sum(errores) / len(errores)

mae_naive = None  # completa con prediccion = y[t−1]
mae_mm3   = None  # completa con prediccion = media de y[t−3..t−1]
skill     = None  # completa: 1 − mae_mm3/mae_naive


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicio 4: folds walk-forward
n_meses, train_inicial, horizonte = 24, 12, 3
folds = []  # completa: [(train_desde, train_hasta, test_desde, test_hasta), ...]


## Reflexión

1. El laboratorio selecciona su umbral usando todos los ejemplos a la vez, sin noción de
   orden temporal. Si esos ejemplos fueran mediciones consecutivas de un sensor, ¿qué
   haría inválida esa selección y cómo la reorganizarías en un esquema walk-forward?
2. ¿Por qué un R² alto prediciendo el nivel de una serie persistente no demuestra nada, y
   qué comparación lo sustituye?
3. Tu backtest da MAE 1.2 y en producción el error es 2.5 desde el primer día. Enumera
   tres causas ordenadas de la más probable a la menos, con el diagnóstico para cada una.
